In [ ]:
# Reciprocal Rank Fusion (RRF) is a simple technique used in RAG/search systems to 
# combine multiple ranked result lists without needing to normalize their scores.
# BM25 scores and vector scores are completely different scales
# Reciprocal means: 1 / rank and Real RRF normally uses a constant such as: k = 60 
"""
For example, user asks:                               "What are the eligibility criteria for EGFR treatment?"
BM25 might find documents containing the exact words:  EGFR treatment eligibility
Vector search might find documents discussing:         patients suitable for targeted EGFR therapy
They may rank documents differently. RRF combines both rankings.

            Rank 1 → 1/(60+1) = 0.01639
            Rank 2 → 1/(60+2) = 0.01613
            Rank 3 → 1/(60+3) = 0.01587

                  User Query
                      │
             ┌────────┴────────┐
             ↓                 ↓
          BM25             Vector Search
             │                 │
             ↓                 ↓
       Ranked List A      Ranked List B
             │                 │
             └────────┬────────┘
                      ↓
                    RRF
                      ↓
              Combined Ranking
                      ↓
                 Top K chunks
                      ↓
                    LLM
"""

In [ ]:
documents = {
    "D1": "Osimertinib common side effects include diarrhea and skin rash.",
    "D2": "Osimertinib is an EGFR tyrosine kinase inhibitor used in lung cancer.",
    "D3": "Patients receiving osimertinib should be monitored for QT prolongation.",
    "D4": "EGFR targeted therapy can cause diarrhea, rash, and nail changes.",
    "D5": "Osimertinib treatment requires monitoring of cardiac function."
}
# We use two retrievers:
# BM25 → looks for matching words
# Vector search → looks for semantic meaning
bm25_results = [
    ("D1", 15.2),
    ("D3", 12.4),
    ("D2", 10.1),
    ("D4", 8.5),
    ("D5", 5.2)
]

vector_results = [
    ("D4", 0.93),
    ("D1", 0.91),
    ("D5", 0.87),
    ("D3", 0.82),
    ("D2", 0.76)
]
#  cannot meaningfully compare above scores of BM25 and Vector score but RRF just interested in ranks 
"""
| Document | BM25 Rank | Vector Rank |
| -------- | --------: | ----------: |
| D1       |         1 |           2 |
| D2       |         3 |           5 |
| D3       |         2 |           4 |
| D4       |         4 |           1 |
| D5       |         5 |           3 |
For example for D1, BM25   → Rank 1 and Vector → Rank 2 so RRF is calculated as follows
D1 = 1/(60+1) + 1/(60+2)
   = 1/61 + 1/62
   = 0.01639 + 0.01613
   = 0.03252
"""
k = 60                   # <<-------This is RRF constant 
rankings = {
    "D1": [1, 2],
    "D2": [3, 5],
    "D3": [2, 4],
    "D4": [4, 1],
    "D5": [5, 3]
}

rrf_scores = {}
for doc, ranks in rankings.items():
    score = 0
    for rank in ranks:
        score += 1 / (k + rank)
    rrf_scores[doc] = score
print(rrf_scores)

final_Ranking = sorted(
    rrf_scores.items(),
    key = lambda x : x[1],
    reverse = False
)
print(final_Ranking)
"""
                    User Query
                        │
              "Osimertinib side effects"
                        │
          ┌─────────────┼─────────────┐
          ↓             ↓             ↓
        BM25         Vector       Metadata
          │             │             │
          ↓             ↓             ↓
      Ranked list   Ranked list   Ranked list
          │             │             │
          └─────────────┼─────────────┘
                        ↓
                       RRF
                        ↓
                Combined Ranking
                        ↓
                  Top 5 documents
                        ↓
                     Reranker
                        ↓
                       LLM
                        ↓
                     Answer
"""
# RRF combines multiple search rankings by giving higher weight to higher-ranked documents, 
# adding their rank-based contributions, and avoiding the need to normalize incompatible search scores.

{'D1': 0.03252247488101534, 'D2': 0.03125763125763126, 'D3': 0.031754032258064516, 'D4': 0.032018442622950824, 'D5': 0.03125763125763126}
[('D2', 0.03125763125763126), ('D5', 0.03125763125763126), ('D3', 0.031754032258064516), ('D4', 0.032018442622950824), ('D1', 0.03252247488101534)]
